# short_volshock_target_generation_handoff_v1

## Goal

The prior official replay screens isolated the short-volatility shock signal: it improves `wf_2022`, near-misses `wf_2023_2024`, and sell-side execution guard tuning does not beat the no-guard official replay. This notebook moves the signal out of runtime execution and into fold-local target generation / checkpoint scoring readiness.

## Governance

- Baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `short_horizon_risk_cap_counterfactual_v1`, `short_volshock_official_replay_screen_v1`, `short_volshock_sell_guard_official_replay_v1`
- Role: `diagnose`
- Changed layer: `selection` / `data` / `model_selection`
- Target windows: `wf_2022_train`, `wf_2023_2024_train` target contracts, with eval windows kept out by scope audit
- Acceptance: generate fold-local short-volshock de-risk target rows, pass target-scope audit, and show BC/checkpoint label distribution is not degenerate
- Promotion: blocked; this is a handoff to checkpoint scoring or BC screen, not an actor or runtime overlay promotion
- Failure mode: no target rows, eval leakage, degenerate action labels, or target semantics that only encode a runtime cap without trainable labels

## External basis

- Qlib official research/backtest framework: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL dynamic trading decision framework: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- DAgger / behavior cloning covariate-shift caution: https://proceedings.mlr.press/v15/ross11a.html
- Risk-sensitive sequential decision framing: https://arxiv.org/abs/1312.4077


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_bc import action_distribution_frame, build_state_objective_bc_dataset_from_frame
from src.soxl_quant.app.state_objective_training_targets import (
    ShortVolshockDeRiskTargetConfig,
    apply_short_volshock_de_risk_target_gate,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_short_volshock_de_risk_targets,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = "short_volshock_target_generation_handoff_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1"
RISK_CARD = ROOT / "runtime/model_selection/card/drawdown_interval_risk_state_coverage_v1"
SELL_GUARD_CARD = ROOT / "runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1"
TARGET_PATH = SOURCE_CARD / "multi_obj_floor45_cap20_riskfirst_target_contract.csv"
FEATURE_PATH = ROOT / "data/soxl_profile_balanced_rule_residual_v22.pkl"
THRESHOLD_PATH = RISK_CARD / "fold_local_signal_thresholds.csv"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SELECTED_CANDIDATE = "multi_obj_floor45_cap20_riskfirst"
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://arxiv.org/abs/1312.4077",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

source_summary = load_json(SOURCE_CARD / "summary.json")
sell_guard_summary = load_json(SELL_GUARD_CARD / "summary.json")
profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGET_PATH, parse_dates=["datetime"])
features = pd.read_pickle(FEATURE_PATH)
thresholds = pd.read_csv(THRESHOLD_PATH)
print("source decision", source_summary["decision"])
print("sell guard decision", sell_guard_summary["decision"])
print("targets", targets.shape, targets["fold"].value_counts().to_dict())


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


source decision state_conditioned_multi_objective_drawdown_failed_no_promotion
sell guard decision official_short_volshock_sell_guard_no_improvement_no_promotion
targets (1009, 46) {'wf_2023_2024_train': 756, 'wf_2022_train': 253}


In [2]:
config = ShortVolshockDeRiskTargetConfig(
    target_cap=0.05,
    min_feature_hits=2,
    min_position_delta=0.02,
    objective_weight_bonus=6.0,
    max_objective_weight=25.0,
)
revised_targets = apply_short_volshock_de_risk_target_gate(
    targets,
    features,
    thresholds,
    config=config,
)
target_summary = summarize_short_volshock_de_risk_targets(revised_targets)

short_cols = [
    "fold",
    "short_volshock_state",
    "short_volshock_target_state",
    "short_volshock_feature_hits",
    "short_volshock_risk_cap_delta",
    "risk_target_position_before_short_volshock",
    "risk_target_position",
    "objective_weight",
]
fold_summary = (
    revised_targets.groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "count"),
        state_rows=("short_volshock_state", "sum"),
        target_state_rows=("short_volshock_target_state", "sum"),
        mean_feature_hits=("short_volshock_feature_hits", "mean"),
        max_risk_cap_delta=("short_volshock_risk_cap_delta", "max"),
        mean_objective_weight=("objective_weight", "mean"),
    )
    .reset_index()
)
revised_targets.to_csv(CARD / "short_volshock_de_risk_targets.csv", index=False)
fold_summary.to_csv(CARD / "short_volshock_target_fold_summary.csv", index=False)
write_json(CARD / "target_summary.json", target_summary)
display(pd.DataFrame([target_summary]))
display(fold_summary)
display(revised_targets.loc[revised_targets["short_volshock_target_state"], short_cols].head(12))


,decision,rows,folds,short_volshock_state_rows,target_state_rows,max_risk_cap_delta,mean_objective_weight,next_mechanism,required_terms
0,short_volshock_de_risk_targets_ready,1009,"[wf_2022_train, wf_2023_2024_train]",477,469,0.775,19.19681,score_checkpoints_or_run_bc_with_short_volshoc...,"[fold_local_short_volshock_state, short_volsho..."


,fold,rows,state_rows,target_state_rows,mean_feature_hits,max_risk_cap_delta,mean_objective_weight
0,wf_2022_train,253,135,133,1.608696,0.775000,25.000000
1,wf_2023_2024_train,756,342,336,1.304233,0.701473,17.254737


,fold,short_volshock_state,short_volshock_target_state,short_volshock_feature_hits,short_volshock_risk_cap_delta,risk_target_position_before_short_volshock,risk_target_position,objective_weight
27,wf_2022_train,True,True,2,0.196684,0.246684,0.05,25.0
28,wf_2022_train,True,True,2,0.196684,0.246684,0.05,25.0
29,wf_2022_train,True,True,2,0.196684,0.246684,0.05,25.0
30,wf_2022_train,True,True,2,0.196684,0.246684,0.05,25.0
31,wf_2022_train,True,True,2,0.196684,0.246684,0.05,25.0
32,wf_2022_train,True,True,3,0.196684,0.246684,0.05,25.0
33,wf_2022_train,True,True,3,0.022464,0.072464,0.05,25.0
34,wf_2022_train,True,True,3,0.022464,0.072464,0.05,25.0
35,wf_2022_train,True,True,3,0.022464,0.072464,0.05,25.0
36,wf_2022_train,True,True,3,0.022464,0.072464,0.05,25.0


In [3]:
fold_defs = []
for fold, group in revised_targets.groupby("fold", sort=True):
    row = group.iloc[0]
    fold_defs.append(
        {
            "name": str(fold),
            "train_start": row["train_start"],
            "train_end": row["train_end"],
            "eval_start": row["eval_start"],
            "eval_end": row["eval_end"],
        }
    )
scope_audit = audit_state_objective_target_scope(revised_targets, folds=fold_defs)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
scope_audit.to_csv(CARD / "target_scope_audit.csv", index=False)
write_json(CARD / "target_scope_summary.json", scope_summary)
display(scope_audit)
display(pd.DataFrame([scope_summary]))


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue,train_start,train_end,eval_start,eval_end
0,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,253,253,0,0,True,,2020-01-02,2021-12-31,2022-01-03,2022-12-30
1,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2020-01-02,2022-12-30,2023-01-03,2024-12-31


,decision,training_allowed,leakage_gate,audited_groups,blocked_groups,eval_overlap_rows,outside_train_eval_rows,scope_issues,top_scope_issue,next_mechanism
0,state_objective_targets_training_ready,True,True,2,0,0,0,[],,run_fold_local_state_objective_training_or_bc


In [4]:
dataset = build_state_objective_bc_dataset_from_frame(
    features,
    targets=revised_targets,
    profile=profile,
    sample_size=0,
    rng_seed=20260526,
)
dist = action_distribution_frame(dataset.actions, action_count=11)
dist.to_csv(CARD / "bc_action_distribution.csv", index=False)
shares = dist["share"].to_numpy(dtype=float)
positive = shares[shares > 0]
action_entropy = float(-(positive * np.log(positive)).sum()) if len(positive) else 0.0
label_comparison = pd.DataFrame(
    [
        {
            "profile": "short_volshock_de_risk_targets",
            "samples": int(len(dataset.actions)),
            "unique_actions": int((dist["count"] > 0).sum()),
            "dominant_action_share": float(dist["share"].max()) if not dist.empty else 1.0,
            "action_entropy": action_entropy,
            "short_volshock_state_samples": int(dataset.metadata.get("short_volshock_state", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
            "short_volshock_target_samples": int(dataset.metadata.get("short_volshock_target_state", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()),
        }
    ]
)
bc_readiness = score_state_objective_bc_readiness(label_comparison, target_scope="fold_local")
bc_summary = summarize_state_objective_bc_readiness(bc_readiness)
label_comparison.to_csv(CARD / "bc_label_comparison.csv", index=False)
bc_readiness.to_csv(CARD / "bc_readiness.csv", index=False)
write_json(CARD / "bc_readiness_summary.json", bc_summary)

slice_distribution = (
    dataset.metadata.assign(label_action=dataset.actions)
    .groupby(["short_volshock_target_state", "label_action"], dropna=False)
    .size()
    .reset_index(name="count")
)
slice_distribution.to_csv(CARD / "short_volshock_label_slice_distribution.csv", index=False)
display(dist)
display(label_comparison)
display(bc_readiness)
display(slice_distribution)


,action,count,share
0,0,230,0.304636
1,1,313,0.414570
2,2,88,0.116556
3,3,19,0.025166
4,4,105,0.139073
5,5,0,0.000000
6,6,0,0.000000
7,7,0,0.000000
8,8,0,0.000000
9,9,0,0.000000


,profile,samples,unique_actions,dominant_action_share,action_entropy,short_volshock_state_samples,short_volshock_target_samples
0,short_volshock_de_risk_targets,755,5,0.41457,1.344684,376,368


,profile,samples,unique_actions,dominant_action_share,action_entropy,short_volshock_state_samples,short_volshock_target_samples,sample_gate,label_distribution_gate,fold_local_scope_gate,bc_readiness_gate,bc_readiness_loss,target_scope
0,short_volshock_de_risk_targets,755,5,0.41457,1.344684,376,368,True,True,True,True,0.0,fold_local


,short_volshock_target_state,label_action,count
0,False,0,73
1,False,1,102
2,False,2,88
3,False,3,19
4,False,4,105
5,True,0,157
6,True,1,211


In [5]:
target_ready = target_summary["decision"] == "short_volshock_de_risk_targets_ready"
scope_ready = bool(scope_summary.get("training_allowed", False))
bc_ready = bool(bc_summary.get("selected_bc_readiness_gate", False))
if target_ready and scope_ready and bc_ready:
    decision = "short_volshock_target_generation_ready_no_promotion"
    next_action = "run a BC/checkpoint-alignment screen using the fold-local short-volshock de-risk targets, then replay wf_2022/wf_2023_2024 before any PPO"
    conclusion = [
        f"Fold-local target generation is viable: {target_summary['target_state_rows']} de-risk rows out of {target_summary['rows']} target rows.",
        f"Scope audit passed with eval_overlap_rows={scope_summary.get('eval_overlap_rows', 0)} and target_scope=fold_local.",
        f"BC labels are not degenerate: samples={bc_summary.get('selected_samples')}, unique_actions={bc_summary.get('selected_unique_actions')}, dominant_share={bc_summary.get('selected_dominant_action_share'):.3f}.",
        "This is not a promotion; it is a handoff to checkpoint scoring / BC warm-start with short-volshock as a trainable risk target.",
        f"Next action: {next_action}.",
    ]
else:
    decision = "short_volshock_target_generation_blocked_no_training"
    next_action = "fix target scope, label degeneracy, or short-volshock state representation before BC/checkpoint scoring"
    conclusion = [
        f"Target generation did not pass all handoff gates: target_ready={target_ready}, scope_ready={scope_ready}, bc_ready={bc_ready}.",
        f"Next action: {next_action}.",
    ]

selection_payload = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention": "none; target-generation handoff only",
    "selected_candidate": SELECTED_CANDIDATE,
    "target_summary": target_summary,
    "scope_summary": scope_summary,
    "bc_summary": bc_summary,
    "next_action": next_action,
}
summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": source_summary.get("canonical_baseline", "v19reg_v22_budget_usage_bull_guard_v1"),
    "direct_comparison_baseline": [
        "short_horizon_risk_cap_counterfactual_v1",
        "short_volshock_official_replay_screen_v1",
        "short_volshock_sell_guard_official_replay_v1",
    ],
    "profile": PROFILE_NAME,
    "role": "diagnose",
    "changed_layer": ["selection", "data", "model_selection"],
    "external_sources": EXTERNAL_SOURCES,
    "closest_historical_success_case": "short_volshock_official_replay_screen_v1 showed short_volshock cap=0.05 min_hits=2 can improve wf_2022 and near-miss wf_2023_2024 under official replay",
    "closest_historical_failure_case": "short_volshock_sell_guard_official_replay_v1 showed runtime sell-side guard tuning increased orders but did not beat no-guard official replay",
    "duplicate_check": "This run does not tune runtime cap thresholds; it converts fold-local short-volshock states into target-generation and BC/checkpoint readiness evidence.",
    "target_summary": target_summary,
    "scope_summary": scope_summary,
    "bc_summary": bc_summary,
    "promotion_allowed": False,
    "decision": decision,
    "conclusion": conclusion,
    "sidecars": [
        "short_volshock_de_risk_targets.csv",
        "short_volshock_target_fold_summary.csv",
        "target_summary.json",
        "target_scope_audit.csv",
        "target_scope_summary.json",
        "bc_action_distribution.csv",
        "bc_label_comparison.csv",
        "bc_readiness.csv",
        "bc_readiness_summary.json",
        "short_volshock_label_slice_distribution.csv",
        "selection.json",
        "summary.json",
    ],
}
write_json(CARD / "selection.json", selection_payload)
write_json(CARD / "summary.json", summary_payload)

conclusion_md = "\n".join(f"- {item}" for item in conclusion)
display(Markdown(f"""
## Conclusion

{conclusion_md}

### Handoff Status

- decision: `{decision}`
- target rows: `{target_summary['rows']}`
- short-volshock state rows: `{target_summary['short_volshock_state_rows']}`
- de-risk target rows: `{target_summary['target_state_rows']}`
- scope training allowed: `{scope_ready}`
- BC readiness: `{bc_ready}`
- promotion: `False`
"""))



## Conclusion

- Fold-local target generation is viable: 469 de-risk rows out of 1009 target rows.
- Scope audit passed with eval_overlap_rows=0 and target_scope=fold_local.
- BC labels are not degenerate: samples=755, unique_actions=5, dominant_share=0.415.
- This is not a promotion; it is a handoff to checkpoint scoring / BC warm-start with short-volshock as a trainable risk target.
- Next action: run a BC/checkpoint-alignment screen using the fold-local short-volshock de-risk targets, then replay wf_2022/wf_2023_2024 before any PPO.

### Handoff Status

- decision: `short_volshock_target_generation_ready_no_promotion`
- target rows: `1009`
- short-volshock state rows: `477`
- de-risk target rows: `469`
- scope training allowed: `True`
- BC readiness: `True`
- promotion: `False`
